# Adım 1 — Ham Videolardan XY (17 bel üstü) + Açı (sağ/sol kol) .npy üretimi
- MediaPipe Pose ile her framede 2D landmark (normalize, [0..1]) çıkar.
- Bel üstü 17 nokta: yüz/omuz/kol + (referans için kalça merkezleri de açı hesabında kullanılacak).
- Açı seti (6): 
  - Sol kol: dirsek(LW-LE-LS), kol-gövde(LE-LS-hip), omuz hattı(LE-LS-midneck) 
  - Sağ kol: dirsek(RW-RE-RS), kol-gövde(RE-RS-hip), omuz hattı(RE-RS-midneck)
- Sonuçları .npy olarak kaydet.


## 1.1 – Yol & Klasörler

In [1]:
import os, glob, json
from pathlib import Path

BASE_ROOT   = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"
WORK_DIR    = os.path.join(BASE_ROOT, "ae_stajdevam")

TRAIN_VID_DIR = BASE_ROOT
TEST_VID_DIR  = os.path.join(BASE_ROOT, "test_video")

OUT_XY_TRAIN   = os.path.join(WORK_DIR, "npy_cikti")
OUT_XY_TEST    = os.path.join(WORK_DIR, "npy_test_cikti")
OUT_ANG_TRAIN  = os.path.join(WORK_DIR, "npy_aci_cikti")
OUT_ANG_TEST   = os.path.join(WORK_DIR, "npy_aci_test_cikti")

for d in [WORK_DIR, OUT_XY_TRAIN, OUT_XY_TEST, OUT_ANG_TRAIN, OUT_ANG_TEST]:
    os.makedirs(d, exist_ok=True)

print("Çıkış klasörleri hazır.")


Çıkış klasörleri hazır.


## 1.2 – Kurulum & İçe Aktarımlar

mediapipe ve opencv-python yoksa ilk satırları aç.

In [2]:
# !pip install mediapipe==0.10.9 opencv-python --quiet

import cv2
import numpy as np
import mediapipe as mp

mp_pose = mp.solutions.pose


## 1.3 – Üst gövde 17 nokta seçimi ve yardımcılar

Omuz-merkez ve omuz genişliği ileride normalizasyon için kritik.

XY .npy’de sadece 17 nokta tutulacak; açı hesabında gerektiğinde kalça mid-point (23/24) ayrıca okunur.

In [3]:
# MediaPipe Pose indeksleri (33 adet)
# Sık kullanılanlar:
#  0:nose, 11:L-shoulder, 12:R-shoulder, 13:L-elbow, 14:R-elbow,
#  15:L-wrist, 16:R-wrist, 23:L-hip, 24:R-hip
# Yüz için: 1..10 (göz/kaş/ağız bağlantıları)
UPPER17 = [0, 1,2,3,4,5,6, 7,8, 9,10, 11,12, 13,14, 15,16]  # toplam 17
LHIP, RHIP = 23, 24
LS, RS, LE, RE, LW, RW = 11, 12, 13, 14, 15, 16  # kısaltmalar

def _angle3(a, b, c):
    """a-b-c açısı (derece). a,b,c: (...,2)"""
    v1 = a - b; v2 = c - b
    v1 /= (np.linalg.norm(v1, axis=-1, keepdims=True) + 1e-8)
    v2 /= (np.linalg.norm(v2, axis=-1, keepdims=True) + 1e-8)
    cosv = np.clip(np.sum(v1*v2, axis=-1), -1.0, 1.0)
    return np.degrees(np.arccos(cosv))

def _fwd_fill_nan(arr):
    """(T,*) NaN'leri öne doğru doldur, başta hepsi NaN ise 0 kabul."""
    arr = arr.copy()
    mask = np.isnan(arr)
    if not mask.any():
        return arr
    # İlk geçerli değeri bul
    for t in range(arr.shape[0]):
        if not np.isnan(arr[t]).any():
            first = arr[t]
            arr[:t] = np.where(np.isnan(arr[:t]), first, arr[:t])
            break
    # İleri doldur
    for t in range(1, arr.shape[0]):
        arr[t] = np.where(np.isnan(arr[t]), arr[t-1], arr[t])
    # Hâlâ NaN kaldıysa 0 yap
    arr = np.nan_to_num(arr, nan=0.0)
    return arr


## 1.4 – Video’dan Landmark Çıkarma (XY) + Açı(6)

In [4]:
def extract_xy_and_angles(video_path: str):
    """
    Dönüş:
      xy_17: (T, 17, 2)  -> [0..1] normalize görüntü koordinatları
      ang6: (T, 6)       -> [deg] sonra /180 ile [0..1] ölçeğine alınabilir
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Açılamadı: {video_path}")

    xy_list = []
    ang_list = []

    with mp_pose.Pose(static_image_mode=False,
                      model_complexity=1,
                      enable_segmentation=False,
                      min_detection_confidence=0.5,
                      min_tracking_confidence=0.5) as pose:

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            # BGR->RGB ve inference
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)
            if not res.pose_landmarks:
                xy_list.append(np.full((len(UPPER17),2), np.nan, dtype=np.float32))
                ang_list.append(np.full((6,), np.nan, dtype=np.float32))
                continue

            lm = res.pose_landmarks.landmark  # 33 adet

            # XY 17 çıkar
            pts17 = []
            for idx in UPPER17:
                pts17.append([lm[idx].x, lm[idx].y])
            pts17 = np.array(pts17, dtype=np.float32)  # (17,2)

            # Açı için gerekli noktalar
            def _p(i): return np.array([lm[i].x, lm[i].y], dtype=np.float32)

            hip = (_p(LHIP) + _p(RHIP)) / 2.0
            midneck = (_p(LS) + _p(RS)) / 2.0

            # Sol kol (LW-LE-LS) dirsek, (LE-LS-hip) kol-gövde, (LE-LS-midneck) omuz hattı
            aL1 = _angle3(_p(LW), _p(LE), _p(LS))
            aL2 = _angle3(_p(LE), _p(LS), hip)
            aL3 = _angle3(_p(LE), _p(LS), midneck)

            # Sağ kol (RW-RE-RS), (RE-RS-hip), (RE-RS-midneck)
            aR1 = _angle3(_p(RW), _p(RE), _p(RS))
            aR2 = _angle3(_p(RE), _p(RS), hip)
            aR3 = _angle3(_p(RE), _p(RS), midneck)

            ang = np.array([aL1, aL2, aL3, aR1, aR2, aR3], dtype=np.float32)  # (6,)

            xy_list.append(pts17)
            ang_list.append(ang)

    cap.release()

    xy_17 = np.stack(xy_list, axis=0) if xy_list else np.zeros((0,17,2), np.float32)
    ang6  = np.stack(ang_list, axis=0) if ang_list else np.zeros((0,6), np.float32)

    # NaN temizliği (takip kaçırma anlarında olur)
    xy_17 = _fwd_fill_nan(xy_17)
    ang6  = _fwd_fill_nan(ang6)

    return xy_17, ang6


## 1.5 – Tek video işle ve kaydet

In [5]:
def process_and_save(video_path: str, out_xy_dir: str, out_ang_dir: str):
    name = Path(video_path).stem  # "1", "2", "61", "63"...
    xy, ang = extract_xy_and_angles(video_path)

    # Kaydet
    xy_path  = os.path.join(out_xy_dir,  f"{name}_xy.npy")
    ang_path = os.path.join(out_ang_dir, f"{name}_ang.npy")
    np.save(xy_path,  xy.astype(np.float32))
    np.save(ang_path, (ang/180.0).astype(np.float32))  # açıları [0,1] ölçekle
    return xy.shape, ang.shape, xy_path, ang_path


## 1.6 – 60 Eğitim + 3 Test videoyu sırayla işle

In [6]:
# Eğitim (1..60).mp4
train_videos = [os.path.join(TRAIN_VID_DIR, f"{i}.mp4") for i in range(1, 61)]
train_videos = [p for p in train_videos if os.path.exists(p)]

# Test (61,62,63).mp4
test_videos = [os.path.join(TEST_VID_DIR, f) for f in ["61.mp4","62.mp4","63.mp4"]]
test_videos = [p for p in test_videos if os.path.exists(p)]

print(f"Eğitim video sayısı: {len(train_videos)}  | Test video sayısı: {len(test_videos)}")

# Çalıştır
report = []
for vp in train_videos:
    shp_xy, shp_ang, xy_p, ang_p = process_and_save(vp, OUT_XY_TRAIN, OUT_ANG_TRAIN)
    report.append(("train", Path(vp).name, shp_xy, shp_ang))

for vp in test_videos:
    shp_xy, shp_ang, xy_p, ang_p = process_and_save(vp, OUT_XY_TEST, OUT_ANG_TEST)
    report.append(("test", Path(vp).name, shp_xy, shp_ang))

print("Tamamlandı. Örnek rapor ilk 5 satır:")
for r in report[:5]:
    print(r)


Eğitim video sayısı: 60  | Test video sayısı: 3
Tamamlandı. Örnek rapor ilk 5 satır:
('train', '1.mp4', (259, 17, 2), (259, 6))
('train', '2.mp4', (273, 17, 2), (273, 6))
('train', '3.mp4', (173, 17, 2), (173, 6))
('train', '4.mp4', (231, 17, 2), (231, 6))
('train', '5.mp4', (197, 17, 2), (197, 6))


## 1.7 – Hızlı kontrol

In [39]:
# Bir örnek yükleyip şekillere bakalım
sample_xy_files = glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy"))
sample_ang_files = glob.glob(os.path.join(OUT_ANG_TRAIN, "*_ang.npy"))
if sample_xy_files and sample_ang_files:
    A_xy = np.load(sample_xy_files[0])
    A_ang = np.load(sample_ang_files[0])
    print("XY shape:", A_xy.shape, "  (T,17,2)")
    print("ANG shape:", A_ang.shape, " (T,6)")
else:
    print("Henüz örnek bulunamadı. Birkaç videoyu işler misin?")


XY shape: (210, 17, 2)   (T,17,2)
ANG shape: (210, 6)  (T,6)


# Adım 2 — Pencereleme (windowing), pad, normalize ve PyTorch Dataset/DataLoader
- XY(17×2) + Açı(6) → her frame için 40 özellik
- XY normalizasyonu: omuz merkezi ile merkezle, omuz genişliğine göre ölçekle
- Pencereleme: window_len=30, stride=15 (pad: edge/zero/reflect)
- Çıkış tensörü: (C=40, T=30)
- Eğitim/Doğrulama ayrımı: video bazlı 80/20


## 2.1 – Konfig (window/pad/normalizasyon)

In [1]:

import os, glob, json, math
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ---- Çalışma klasörleri (bir önceki adımla aynı yapıda) ----
BASE_ROOT   = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"   # projenin temel klasörü
WORK_DIR    = os.path.join(BASE_ROOT, "ae_stajdevam")                  # bu çalışmaya özel alt klasör

# Eğitim ve test için XY ve Açı (angle) çıktılarının yolları
OUT_XY_TRAIN   = os.path.join(WORK_DIR, "npy_cikti")           # eğitim XY .npy dosyaları
OUT_XY_TEST    = os.path.join(WORK_DIR, "npy_test_cikti")      # test XY .npy dosyaları
OUT_ANG_TRAIN  = os.path.join(WORK_DIR, "npy_aci_cikti")       # eğitim açı .npy dosyaları
OUT_ANG_TEST   = os.path.join(WORK_DIR, "npy_aci_test_cikti")  # test açı .npy dosyaları

# Özellik normalizasyon istatistik dosyası (mean/std)
STATS_JSON     = os.path.join(WORK_DIR, "feat_stats.json")

# ---- Deney/ön işleme yapılandırması ----
CFG = {
    "window_len": 30,        # her pencerenin kare (frame) uzunluğu
    "stride": 15,            # pencere kaydırma adımı (overlap=window_len - stride)
    "pad_mode": "edge",      # kırpma dışında kalan kısım için doldurma: "edge" | "zero" | "reflect"
    "use_angles": True,      # True ise 6 açıyı XY ile birleştir (toplam 40 kanal varsayımı)
    "batch_size": 128,       # encode/eğitim sırasında mini-batch boyutu
    "num_workers": 0,        # DataLoader işçi sayısı (Jupyter için 0 daha stabil)
    "seed": 42               # tekrar üretilebilirlik için rastgelelik tohumu
}

# ---- Deterministiklik için seed ayarları ----
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

# Yapılandırmayı ekrana yaz
print("CFG:", CFG)


CFG: {'window_len': 30, 'stride': 15, 'pad_mode': 'edge', 'use_angles': True, 'batch_size': 128, 'num_workers': 0, 'seed': 42}


## 2.2 – XY normalizasyonu ve özellik birleştirme (XY + Açı)

In [2]:

# UPPER17 sırasıyla kaydetmiştik: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]
# Bu dizide 11 -> L-Shoulder (sol omuz), 12 -> R-Shoulder (sağ omuz) pozisyon indeksleri
LS_POS, RS_POS = 11, 12

def normalize_xy_by_shoulders(xy_17: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Omuz merkezine göre çevirisel normalizasyon + omuz genişliğine göre ölçekleme.
    xy_17: (T,17,2)  -> koordinatlar [0..1] aralığında varsayılır.
    Dönüş: (T,17,2)  -> her kare için omuz-merk ezli ve omuz mesafesine bölünmüş koordinatlar.
    """
    # Omuz merkezi: sol ve sağ omuz noktalarının ortalaması -> (T,2)
    c = (xy_17[:, LS_POS, :] + xy_17[:, RS_POS, :]) / 2.0

    # Omuz genişliği: iki omuz arasındaki Öklid mesafe -> (T,1)
    s = np.linalg.norm(xy_17[:, RS_POS, :] - xy_17[:, LS_POS, :], axis=1, keepdims=True)

    # Çok küçük değerlerde bölme hatasını önlemek için taban değer uygula
    s = np.maximum(s, eps)

    # Merkezle (c çıkar) ve ölçekle (s'ye böl) -> (T,17,2)
    xy_n = (xy_17 - c[:, None, :]) / s[:, None, None]

    # Tip güvenliği: float32
    return xy_n.astype(np.float32)

def build_frame_features(xy_17: np.ndarray, ang6: np.ndarray, use_angles: bool = True) -> np.ndarray:
    """
    Kare-bazlı özellik vektörü oluşturma.
    Girdi:
      - xy_17: (T,17,2)  -> 17 üst vücut eklemi için (x,y)
      - ang6 : (T,6)     -> 6 açısal özellik (opsiyonel)
      - use_angles: True ise açılar da birleştirilir.
    Dönüş:
      - (C, T) tensörü; C = 34 (+6) = 40 (açı kullanılıyorsa), aksi halde 34
    """
    # Omuz-normalize edilmiş koordinatlar -> (T,17,2)
    xy_n = normalize_xy_by_shoulders(xy_17)

    # (T,17,2) -> (T,34) düzleştirme (her kare için 34 özellik)
    xy_flat = xy_n.reshape(xy_n.shape[0], -1)

    # Açıları kullan: (T,34) ile (T,6)'yı kanallar ekseninde birleştir -> (T,40)
    if use_angles and ang6 is not None and ang6.size > 0:
        feats = np.concatenate([xy_flat, ang6], axis=1)  # (T,40)
    else:
        feats = xy_flat                                   # (T,34)

    # (T,C) -> (C,T) transpoz + tip dönüşümü
    return feats.T.astype(np.float32)                     # (C,T)


## 2.3 – Pencereleme ve pad

In [3]:

def window_and_pad(x: np.ndarray, win: int, stride: int, mode: str = "edge") -> np.ndarray:
    """
    Zaman serisini sabit uzunluklu pencerelere böler, eksik kısımları pad ederek tamamlar.
    Girdi:
      - x  : (C, T)   C: kanal sayısı, T: zaman uzunluğu
      - win: pencere uzunluğu (frame sayısı)
      - stride: pencere kaydırma adımı
      - mode: pad etme biçimi -> "edge" | "zero" | "reflect"
    Çıktı:
      - (Nwin, C, win)  Nwin: oluşturulan pencere sayısı
    """
    C, T = x.shape
    out = []

    # T <= 0 ise en az bir pencere döndür (tamamen sıfır pad'li)
    if T <= 0:
        return np.zeros((1, C, win), dtype=x.dtype)

    # 0..(T-win) aralığında stride adımıyla pencereler üret
    # max(1, T-win+1): T < win ise en az bir kez döngüye girsin diye
    for start in range(0, max(1, T - win + 1), stride):
        end = start + win
        if end <= T:
            # Tam sığan pencere: direkt dilimle
            out.append(x[:, start:end])
        else:
            # Son pencere taşarsa, eksik kısmı pad ederek tamamla
            deficit = end - T
            if mode == "zero":
                # Sıfırlarla doldur
                pad = np.zeros((C, deficit), dtype=x.dtype)
            elif mode == "reflect":
                # Son kısımdan ayna yansımalı pad
                take = min(deficit, T)
                pad = np.flip(x[:, T - take:T], axis=1)
                # Eksik pad hâlâ varsa, en baştaki ilk değeri tekrar ederek tamamla
                if take < deficit:
                    pad = np.concatenate([pad, np.repeat(x[:, :1], deficit - take, axis=1)], axis=1)
            else:  # "edge" (varsayılan): son örneği kopyalayarak doldur
                pad = np.repeat(x[:, -1:], deficit, axis=1)
            # Taşan pencereyi, mevcut kısım + pad ile birleştir
            out.append(np.concatenate([x[:, start:T], pad], axis=1))

    # Hiç pencere eklenmediyse (ör. T < win ve for döngüsüne girilemediyse)
    if not out:
        deficit = win - T
        if mode == "zero":
            pad = np.zeros((C, deficit), dtype=x.dtype)
        elif mode == "reflect":
            take = min(deficit, T)
            pad = np.flip(x[:, T - take:T], axis=1)
            if take < deficit:
                pad = np.concatenate([pad, np.repeat(x[:, :1], deficit - take, axis=1)], axis=1)
        else:  # "edge"
            pad = np.repeat(x[:, -1:], deficit, axis=1)
        out = [np.concatenate([x, pad], axis=1)]

    return np.stack(out, axis=0)  # (Nwin, C, win)


## 2.4 – Eğitim seti için özellik istatistikleri (mean/std) çıkar ve kaydet

In [4]:
def compute_and_save_feature_stats(xy_dir: str, ang_dir: str, stats_json: str, cfg: dict):
    """
    Tüm eğitim videolarını dolaşarak (frame bazlı) ortalama ve std hesaplar.
    Not: Önce frame özellikleri -> sonra (C,) mean/std
    """
    # XY koordinat .npy dosyalarını (…*_xy.npy) topla ve isme göre sırala
    xy_files = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
    # Eğitim verisi yoksa süreci durdur (hızlı hata yakalama)
    assert xy_files, f"Eğitim XY bulunamadı: {xy_dir}"
    # Kümülatif toplam vektörleri (kanal bazlı) — ilk turda boyut bilinmediği için None
    sums = None
    sums2 = None
    # Toplam frame sayacı (T’lerin toplamı)
    count = 0

    # Her bir XY dosyası için döngü
    for xy_path in xy_files:
        # Dosya taban adını çıkart (…_xy.npy uzantısını kaldır)
        base = Path(xy_path).name.replace("_xy.npy", "")
        # Açı dosyasının yolu (varsa); isim eşlemesi: <base>_ang.npy
        ang_path = os.path.join(ang_dir, f"{base}_ang.npy")
        # XY: (T,17,2) — T kare sayısı, 17 eklem, (x,y)
        xy = np.load(xy_path)            # (T,17,2)
        # ANG: (T,6) [0..1] — açı/özellik vektörü; yoksa None
        ang = np.load(ang_path) if os.path.exists(ang_path) else None  # (T,6) [0..1]

        # Frame başına özellik inşası: çıktı (C,T) olacak şekilde kanal x zaman
        # cfg["use_angles"] True ise açı kanalları da eklenir
        feat = build_frame_features(xy, ang, cfg["use_angles"])  # (C,T)
        C, T = feat.shape  # C: kanal sayısı, T: frameler

        # İlk dosyada kümülatif konteynerleri (C,) boyutunda oluştur
        if sums is None:
            sums  = np.zeros(C, dtype=np.float64)
            sums2 = np.zeros(C, dtype=np.float64)

        # Ağırlıklı toplamlar:
        # feat.mean(axis=1): her kanal için bu dosyanın ortalaması
        # T ile çarparak tüm framelere göre ağırlıklandır (dosyalar arası farklı T’lere adil)
        sums  += feat.mean(axis=1) * T    # ortalamanın T ile ağırlığı
        # İkinci moment (x^2) ortalaması da aynı şekilde ağırlıklandırılır
        sums2 += (feat**2).mean(axis=1) * T
        # Toplam frame sayısını artır
        count += T

    # Nihai kanal ortalaması: (toplam ağırlıklı ortalama) / (toplam frame)
    mean = (sums / count).astype(np.float32)
    # Varyans: E[x^2] - (E[x])^2 formülü; sayısal kararlılık için float64’te kare
    var  = (sums2 / count) - (mean.astype(np.float64)**2)
    # Negatif yuvarlanma hatalarını engellemek için max(var, 1e-8), sonra std
    std  = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)

    # JSON’a yazmak için sözlük: mean/std list’e çevrilir, C kanal sayısı meta olarak eklenir
    stats = {"mean": mean.tolist(), "std": std.tolist(), "C": int(len(mean))}
    # İstatistikleri dosyaya kaydet
    with open(stats_json, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)
    # Kayıt bilgisi
    print(f"Kaydedildi: {stats_json}")
    # Eğitim/çıkarımda doğrudan kullanılabilmesi için numpy array olarak da döndür
    return mean, std

def load_feature_stats(stats_json: str):
    # Kaydedilmiş istatistikleri JSON’dan oku
    with open(stats_json, "r", encoding="utf-8") as f:
        stats = json.load(f)
    # Liste -> numpy array dönüşümü; tipler eğitimde kullanılanla uyumlu (float32)
    mean = np.array(stats["mean"], dtype=np.float32)
    std  = np.array(stats["std"], dtype=np.float32)
    # Kullanıma hazır mean/std döndür
    return mean, std


## 2.5 – Global indeksleme ile PyTorch Dataset

In [5]:
def count_windows(T: int, win: int, stride: int) -> int:
    # Pencere sayısını hesaplar (son pencere kenara yapışır; eksik kısım pad'lenir).
    if T <= 0:
        # Boş/düzensiz veri durumunda en az 1 örnek döndür
        return 1
    if T <= win:
        # T, pencere uzunluğundan küçük/eşitse yalnızca 1 pencere yeterli
        return 1
    # İlk pencere sabit: [0, win). Kalan kısım stride ile kapatılır.
    # (T - win) aralığı stride'a bölünür; tam bölünmüyorsa yukarı yuvarlanır (ceil).
    return 1 + math.ceil((T - win) / stride)

class PoseWindowDataset(Dataset):
    """
    Global indeks → (file_idx, start_index) haritalı.
    __getitem__ dönünce: (C,win) tensörü.
    """
    def __init__(self, xy_dir: str, ang_dir: str, stats_json: str, cfg: dict, file_list=None):
        # Dizin/ayarları sakla
        self.xy_dir = xy_dir
        self.ang_dir = ang_dir
        self.cfg = cfg

        # Tüm XY dosyalarını topla (…*_xy.npy)
        all_xy = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
        if file_list is not None:
            # Eğer belirli bir dosya listesi verildiyse sadece onları kullan
            # file_list: baz adlar (örn: "1", "2", ...); gerçek yola dönüştür
            all_xy = [os.path.join(xy_dir, f"{b}_xy.npy") for b in file_list]
        # Boşsa erken hata ver
        assert all_xy, f"Boş dizin: {xy_dir}"

        # Meta yapılar
        self.files = []   # Her giriş: (xy_path, ang_path, T)
        self.index = []   # Global indeks → (file_idx, start_frame)
        win, stride = cfg["window_len"], cfg["stride"]

        # Normalizasyon istatistiklerini yükle (kanal bazlı mean/std)
        self.mean, self.std = load_feature_stats(stats_json)
        self.C = len(self.mean)  # kanal sayısı (kontrol amaçlı)

        # Her dosya için pencere başlangıç indekslerini oluştur
        for xy_path in all_xy:
            # Baz adı bul (…_xy.npy → baz)
            base = Path(xy_path).name.replace("_xy.npy", "")
            # Açı dosyası yolu (varsa)
            ang_path = os.path.join(ang_dir, f"{base}_ang.npy")
            # Verileri yükle
            xy = np.load(xy_path)
            ang = np.load(ang_path) if os.path.exists(ang_path) else None

            # Frame başına özellikleri üret (çıktı (C,T))
            feats = build_frame_features(xy, ang, cfg["use_angles"])  # (C,T)
            T = feats.shape[1]
            # Bu dosyadan kaç pencere çıkacağını hesapla
            nwin = count_windows(T, win, stride)
            # Dosya kaydını ekle
            self.files.append((xy_path, ang_path, T))

            # Her pencere için başlangıç frame indeksini yaz
            # Not: Slicing gerçek zamanda window_and_pad mantığına göre yapılır
            for k in range(nwin):
                # Kenara yapıştırma: son pencere (T - win) den başlatılır
                start = min(k*stride, max(0, T - win))
                self.index.append((len(self.files)-1, start))

        # Özet bilgi
        print(f"Toplam video: {len(self.files)} | Toplam pencere: {len(self.index)}")

    def __len__(self):
        # Toplam örnek sayısı = üretilen pencere sayısı
        return len(self.index)

    def __getitem__(self, idx):
        # Global indeks → ilgili dosya ve pencere başlangıcı
        file_idx, start = self.index[idx]
        xy_path, ang_path, T = self.files[file_idx]
        # Ham verileri tekrar yükle (her çağrıda)
        xy = np.load(xy_path)
        ang = np.load(ang_path) if os.path.exists(ang_path) else None

        # (C,T) özellik matrisi oluştur
        xCT = build_frame_features(xy, ang, self.cfg["use_angles"])  # (C,T)
        # Kanal bazlı standardizasyon: (x - mean) / std
        xCT = (xCT - self.mean[:, None]) / (self.std[:, None] + 1e-8)
        # Pencere boyu ve kaydırma
        win = self.cfg["window_len"]; stride = self.cfg["stride"]
        stop = start + win

        if stop <= xCT.shape[1]:
            # Tam pencere sığıyorsa direkt dilimle
            xwin = xCT[:, start:stop]
        else:
            # Sondaysa ve yetmiyorsa tek örneklik pad uygula
            deficit = stop - xCT.shape[1]

            if self.cfg["pad_mode"] == "zero":
                # Sıfır dolgu
                pad = np.zeros((xCT.shape[0], deficit), dtype=xCT.dtype)

            elif self.cfg["pad_mode"] == "reflect":
                # Yansıma dolgusu: son kısımdan ters çevirerek al
                take = min(deficit, xCT.shape[1])
                pad = np.flip(xCT[:, -take:], axis=1)
                if take < deficit:
                    # Hâlâ eksik varsa son kolonu tekrar et
                    pad = np.concatenate([pad, np.repeat(xCT[:, -1:], deficit - take, axis=1)], axis=1)
            else:
                # Varsayılan: son kolonu tekrar etme dolgusu
                pad = np.repeat(xCT[:, -1:], deficit, axis=1)

            # Pencere: kalan + pad
            xwin = np.concatenate([xCT[:, start:], pad], axis=1)

        # PyTorch tensörüne çevir ve tip dönüştür (float32)
        x = torch.from_numpy(xwin.astype(np.float32))   # (C,win)
        return x


## 2.6 – Eğitim/Doğrulama ayrımı ve DataLoader’lar

In [6]:
# ==== 2.7 — Eğitim/Doğrulama ayrımı + İstatistik + Dataset/DataLoader (temiz & bağımsız) ====
import os, glob, json, math
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ------------------ Yollar & Ayarlar ------------------
BASE_ROOT = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"
WORK_DIR  = os.path.join(BASE_ROOT, "ae_stajdevam")
OUT_XY_TRAIN  = os.path.join(WORK_DIR, "npy_cikti")
OUT_ANG_TRAIN = os.path.join(WORK_DIR, "npy_aci_cikti")
STATS_JSON    = os.path.join(WORK_DIR, "feat_stats.json")

CFG = {"window_len":30, "stride":15, "pad_mode":"edge",
       "use_angles":True, "batch_size":128, "num_workers":0, "seed":42}
np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

# ------------------ Yardımcılar (bu hücreye özel) ------------------
LS_POS, RS_POS = 11, 12  # L-Shoulder, R-Shoulder

def _normalize_xy_by_shoulders(xy_17: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    # xy_17: (T,17,2) -> omuz merkezle/ölçekle
    xy_17 = np.asarray(xy_17, dtype=np.float32)
    assert xy_17.ndim == 3 and xy_17.shape[1:] == (17,2), f"XY (T,17,2) olmalı, geldi {xy_17.shape}"
    T = xy_17.shape[0]
    # center (T,2), scale (T,1)
    c = (xy_17[:, LS_POS, :] + xy_17[:, RS_POS, :]) / 2.0
    s = np.linalg.norm(xy_17[:, RS_POS, :] - xy_17[:, LS_POS, :], axis=1, keepdims=True)
    s = np.maximum(s, eps)
    # force explicit shapes for safe broadcasting
    c = c.reshape(T, 1, 2)   # (T,1,2)
    s = s.reshape(T, 1, 1)   # (T,1,1)
    return ((xy_17 - c) / s).astype(np.float32)

def _make_features_CT(xy_17: np.ndarray, ang6: np.ndarray | None, use_angles=True) -> np.ndarray:
    """
    Girdi: xy_17 (T,17,2), ang6 (T,6) veya None
    Çıktı: (C,T) -> C=40 (34 XY + 6 açı)
    """
    xy_17 = np.asarray(xy_17, dtype=np.float32)
    assert xy_17.ndim == 3 and xy_17.shape[1:] == (17,2), f"XY (T,17,2) olmalı, geldi {xy_17.shape}"
    T = xy_17.shape[0]

    if ang6 is None:
        ang6 = np.zeros((T,6), dtype=np.float32)
    else:
        ang6 = np.asarray(ang6, dtype=np.float32)
        if ang6.ndim == 1 and ang6.size % 6 == 0:
            ang6 = ang6.reshape(-1,6)
        if ang6.shape == (6, T):
            ang6 = ang6.T
        assert ang6.ndim == 2 and ang6.shape[1] == 6, f"Açı (T,6) olmalı, geldi {ang6.shape}"
        if ang6.shape[0] != T:
            out = np.zeros((T,6), np.float32)
            tmin = min(T, ang6.shape[0])
            out[:tmin] = ang6[:tmin]
        ang6 = out if 'out' in locals() else ang6

    xy_n    = _normalize_xy_by_shoulders(xy_17)  # (T,17,2)
    xy_flat = xy_n.reshape(T, 34)                # (T,34)
    feats_TxC = np.concatenate([xy_flat, ang6], axis=1) if use_angles else xy_flat  # (T,40/34)
    feats_CxT = feats_TxC.T.astype(np.float32)  # (C,T)
    if use_angles:
        assert feats_CxT.shape[0] == 40, f"Kanal 40 olmalı, geldi {feats_CxT.shape[0]}"
    else:
        assert feats_CxT.shape[0] == 34, f"Kanal 34 olmalı, geldi {feats_CxT.shape[0]}"
    return feats_CxT

def _window_slice_or_pad(xCT: np.ndarray, start: int, win: int, mode: str="edge") -> np.ndarray:
    C, T = xCT.shape
    end = start + win
    if end <= T:
        return xCT[:, start:end]
    deficit = end - T
    if mode == "zero":
        pad = np.zeros((C, deficit), dtype=xCT.dtype)
    elif mode == "reflect":
        take = min(deficit, T)
        pad = np.flip(xCT[:, -take:], axis=1)
        if take < deficit:
            pad = np.concatenate([pad, np.repeat(xCT[:, -1:], deficit - take, axis=1)], axis=1)
    else:  # edge
        pad = np.repeat(xCT[:, -1:], deficit, axis=1)
    return np.concatenate([xCT[:, start:], pad], axis=1)

def _load_feature_stats(stats_json: str):
    with open(stats_json, "r", encoding="utf-8") as f:
        s = json.load(f)
    mean = np.array(s["mean"], np.float32)
    std  = np.array(s["std"],  np.float32)
    return mean, std

# ------------------ 1) İstatistik: yoksa hesapla ------------------
if not os.path.exists(STATS_JSON):
    xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))
    assert xy_files, f"Eğitim XY bulunamadı: {OUT_XY_TRAIN}"
    sums  = np.zeros(40, np.float64); sums2 = np.zeros(40, np.float64); count = 0
    for p in xy_files:
        base  = Path(p).name.replace("_xy.npy", "")
        p_ang = os.path.join(OUT_ANG_TRAIN, f"{base}_ang.npy")
        xy = np.load(p)                                     # (T,17,2)
        ang = np.load(p_ang) if os.path.exists(p_ang) else None  # (T,6) veya None
        feat = _make_features_CT(xy, ang, CFG["use_angles"])     # (40,T)
        sums  += feat.sum(axis=1)
        sums2 += (feat**2).sum(axis=1)
        count += feat.shape[1]
    mean = (sums / count).astype(np.float32)
    var  = (sums2 / count) - (mean.astype(np.float64)**2)
    std  = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)
    with open(STATS_JSON, "w", encoding="utf-8") as f:
        json.dump({"mean":mean.tolist(), "std":std.tolist(), "C":40}, f, indent=2)
print("feat_stats.json:", STATS_JSON, "->", os.path.exists(STATS_JSON))

# ------------------ 2) Split (80/20 video bazlı) ------------------
all_train_xy = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))
assert all_train_xy, f"Eğitim XY bulunamadı: {OUT_XY_TRAIN}"
base_names = [Path(p).name.replace("_xy.npy", "") for p in all_train_xy]
base_names = sorted(base_names, key=lambda s: int(Path(s).stem) if Path(s).stem.isdigit() else s)
n = len(base_names); n_train = max(1, int(0.8*n))
train_list = base_names[:n_train]
val_list   = base_names[n_train:] if n_train < n else base_names[-1:]  # en az 1 val
print(f"Video bazlı split -> Train: {len(train_list)} | Val: {len(val_list)} | Toplam: {n}")

# ------------------ 3) Dataset ------------------
class PoseWindowDataset(Dataset):
    def __init__(self, xy_dir: str, ang_dir: str, stats_json: str, cfg: dict, file_list=None):
        self.cfg = cfg
        self.mean, self.std = _load_feature_stats(stats_json)
        xy_files = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
        if file_list is not None:
            xy_files = [os.path.join(xy_dir, f"{b}_xy.npy") for b in file_list]
        assert xy_files, f"Boş dizin: {xy_dir}"

        self.items = []  # (feat(40,T), base)
        self.index = []  # (i, start)
        for p in xy_files:
            base  = Path(p).name.replace("_xy.npy", "")
            p_ang = os.path.join(ang_dir, f"{base}_ang.npy")
            xy = np.load(p)
            ang = np.load(p_ang) if os.path.exists(p_ang) else None
            feat = _make_features_CT(xy, ang, cfg["use_angles"])  # (40,T)
            self.items.append((feat, base))

            T = feat.shape[1]; win = cfg["window_len"]; stride = cfg["stride"]
            if T <= win:
                self.index.append((len(self.items)-1, 0))
            else:
                for st in range(0, T - win + 1, stride):
                    self.index.append((len(self.items)-1, st))
                if (T - win) % stride != 0:
                    self.index.append((len(self.items)-1, T - win))
        print(f"Toplam video: {len(self.items)} | Toplam pencere: {len(self.index)}")

    def __len__(self): return len(self.index)

    def __getitem__(self, idx):
        i, start = self.index[idx]
        feat, _ = self.items[i]  # (40,T)
        x = _window_slice_or_pad(feat, start, self.cfg["window_len"], self.cfg["pad_mode"])
        # standardizasyon
        x = (x - self.mean[:,None]) / (self.std[:,None] + 1e-8)
        return torch.from_numpy(x.astype(np.float32))  # (40,win)

# ------------------ 4) DataLoader'lar ------------------
train_ds = PoseWindowDataset(OUT_XY_TRAIN, OUT_ANG_TRAIN, STATS_JSON, CFG, file_list=train_list)
val_ds   = PoseWindowDataset(OUT_XY_TRAIN, OUT_ANG_TRAIN, STATS_JSON, CFG, file_list=val_list)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], drop_last=False)

# ------------------ 5) Hızlı kontrol ------------------
xb = next(iter(train_loader))
print("Bir batch şekli:", tuple(xb.shape))  # (B, 40, 30) beklenir


feat_stats.json: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\feat_stats.json -> True
Video bazlı split -> Train: 48 | Val: 12 | Toplam: 60
Toplam video: 48 | Toplam pencere: 624
Toplam video: 12 | Toplam pencere: 140
Bir batch şekli: (128, 40, 30)


In [7]:
import glob, os, numpy as np

# Çıktı dizinini ekrana yaz (hızlı doğrulama için)
print('OUT_XY_TRAIN =', OUT_XY_TRAIN)

# OUT_XY_TRAIN içindeki *_xy.npy dosyalarını topla ve sırala
xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))

# Kaç dosya bulunduğunu yazdır
print('found', len(xy_files), 'xy files')

# İlk 8 dosyanın temel bilgilerini göster (örnekleme amaçlı)
for i,p in enumerate(xy_files[:8]):
    print(i, p)
    try:
        # Numpy dizisini yükle
        a = np.load(p)
        # Boyut (shape), veri tipi (dtype), eleman sayısı (size), boyut sayısı (ndim) bilgilerini yaz
        print('  -> shape', a.shape, 'dtype', a.dtype, 'size', a.size, 'ndim', a.ndim)
    except Exception as e:
        # Yükleme hatasını yakala ve mesajı yazdır
        print('  -> load error', e)

# Jupyter/Kernel içinde en son kullanılan 'p' değişkeni mevcutsa onunla tekrar dene
if 'p' in globals():
    print('\nlast p variable from kernel:', p)
    try:
        # Son 'p' yolundaki dosyayı tekrar yükleyip ayrıntılarını göster
        a = np.load(p)
        print('  last p shape', a.shape, 'dtype', a.dtype, 'size', a.size, 'ndim', a.ndim)
    except Exception as e:
        # Yüklemede hata olursa burada raporla
        print('  error loading last p:', e)


OUT_XY_TRAIN = C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti
found 60 xy files
0 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\10_xy.npy
  -> shape (210, 17, 2) dtype float32 size 7140 ndim 3
1 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\11_xy.npy
  -> shape (195, 17, 2) dtype float32 size 6630 ndim 3
2 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\12_xy.npy
  -> shape (214, 17, 2) dtype float32 size 7276 ndim 3
3 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\13_xy.npy
  -> shape (222, 17, 2) dtype float32 size 7548 ndim 3
4 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\14_xy.npy
  -> shape (239, 17, 2) dtype float32 size 8126 ndim 3
5 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\15_xy.npy
  -> shape (194, 17, 2) dtype float32 size 6596 ndim 3
6 C:\Users\The Coder Farmer\Desktop\Auto

In [8]:
import glob, os, numpy as np

# OUT_XY_TRAIN dizininde bulunan tüm *_xy.npy dosyalarını sırala
xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))

# Hatalı veya beklenmeyen dosyaları kaydetmek için liste
bad = []

# Her dosyayı sırayla kontrol et
for i, p in enumerate(xy_files):
    try:
        # Dosyayı yüklemeyi dene
        a = np.load(p)
    except Exception as e:
        # Yükleme başarısızsa hatayı listeye ekle (yol, hata türü, hata mesajı)
        bad.append((p, 'load_error', str(e)))
        continue

    # Beklenen şekil (ndim = 3 ve shape[1:] = (17,2)) mi kontrol et
    ok = (a.ndim == 3 and a.shape[1:] == (17, 2))
    if not ok:
        # Şekil uyumsuzsa hatalılar listesine ekle (yol, hata türü, shape bilgisi, eleman sayısı, boyut sayısı)
        bad.append((p, 'shape', a.shape, a.size, a.ndim))

# Özet: kaç dosya incelendi ve kaç tanesi hatalı
print('checked', len(xy_files), 'files, bad count =', len(bad))

# Hatalı dosyaların ayrıntılarını yazdır
for b in bad:
    print(b)

# Boyut istatistikleri 
from collections import Counter

# Her dosyanın toplam eleman sayısına göre (a.size) istatistik topla
sizes = Counter()
for p in xy_files:
    a = np.load(p)
    sizes[a.size] += 1

# Kaç farklı boyut unique size olduğunu 
print('\nunique sizes count:', len(sizes))

# En sık görülen 10 farklı size ve bunların kaç dosyada bulunduğu
for s, cnt in sizes.most_common(10):
    print(s, cnt)


checked 60 files, bad count = 0

unique sizes count: 45
7242 4
7140 3
6630 3
7004 3
6460 3
6256 3
8806 2
6426 2
7276 1
7548 1


In [9]:
import glob, os, numpy as np

# OUT_XY_TRAIN altındaki tüm *_xy.npy dosyalarını alfabetik sırayla topla
xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))

# İlk 6 dosyada hızlı sağlık kontrolü yap
for p in xy_files[:6]:
    print('\nFILE:', p)
    # Dosyayı yükle (beklenen: (T, 17, 2) — T: frame sayısı, 17 eklem, 2 koordinat)
    a = np.load(p)
    print(' raw shape:', a.shape, 'size', a.size, 'ndim', a.ndim)

    try:
        # T: zaman eksenindeki frame sayısı
        T = a.shape[0]

        # Omuzlara göre normalize et (örn. omuz genişliğiyle ölçekleme + hizalama)
        # Not: _normalize_xy_by_shoulders kullanıcı tanımlı bir fonksiyon olmalı
        xy_n = _normalize_xy_by_shoulders(a)
        print(' after normalize shape:', xy_n.shape, 'size', xy_n.size, 'ndim', xy_n.ndim)

        try:
            # (T, 17, 2) → (T, 34) düzleştirme: her frame için 34 özellik (x1,y1,...,x17,y17)
            xy_flat = xy_n.reshape(T, 34)
            print(' reshape OK ->', xy_flat.shape)
        except Exception as e:
            # Şekil beklenenden farklıysa veya T tutmuyorsa reshape hatası
            print(' reshape ERROR:', e)

    except Exception as e:
        # Normalizasyon öncesinde veya sırasında beklenmedik hata olursa
        print(' normalize ERROR:', e)



FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\10_xy.npy
 raw shape: (210, 17, 2) size 7140 ndim 3
 after normalize shape: (210, 17, 2) size 7140 ndim 3
 reshape OK -> (210, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\11_xy.npy
 raw shape: (195, 17, 2) size 6630 ndim 3
 after normalize shape: (195, 17, 2) size 6630 ndim 3
 reshape OK -> (195, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\12_xy.npy
 raw shape: (214, 17, 2) size 7276 ndim 3
 after normalize shape: (214, 17, 2) size 7276 ndim 3
 reshape OK -> (214, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\13_xy.npy
 raw shape: (222, 17, 2) size 7548 ndim 3
 after normalize shape: (222, 17, 2) size 7548 ndim 3
 reshape OK -> (222, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\14_xy.npy
 raw shape: (239, 17, 2) size 8126 ndim 3
 after normali

In [10]:
import numpy as np, glob, os

# OUT_XY_TRAIN içindeki ilk *_xy.npy dosyasını seç
p = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))[0]
print('file', p)

# XY dizisini yükle (beklenen şekil: (T, 17, 2))
xy = np.load(p)
print('xy shape', xy.shape, 'dtype', xy.dtype)

# Güvenli olması için float32 tipine çevir (kopya veya view olabilir)
xy_17 = np.asarray(xy, dtype=np.float32)
print('xy_17 shape', xy_17.shape, 'ndim', xy_17.ndim)

# Omuz indeksleri: LS = Left Shoulder, RS = Right Shoulder
LS_POS, RS_POS = 11,12

# Sol ve sağ omuz koordinatlarını (T, 2) olarak ayıkla
left = xy_17[:, LS_POS, :]
right = xy_17[:, RS_POS, :]
print('left', left.shape, 'right', right.shape)

# Omuzların orta noktası (merkez) c: (T, 2)
c = (left + right) / 2.0

# Omuzlar arası mesafe (omuz genişliği) s: (T, 1)
# np.linalg.norm(..., axis=1) -> her frame için Öklid normu
s = np.linalg.norm(right - left, axis=1, keepdims=True)
print('c', c.shape, 's', s.shape)

# Yaygın broadcast kontrolü: (T,1,2) ve (T,1,1) şekillerini yazdır
print('c[:,None,:]', c[:,None,:].shape)
print('s[:,None,None]', s[:,None,None].shape)

# Merkezden çıkar: tüm eklemlerden omuz orta noktası c çıkarılır → orijin kaydırma
# Sonuç şekil (T, 17, 2)
res = (xy_17 - c[:, None, :])
print('after subtract shape', res.shape)

# Ölçekle: omuz genişliğine böl (boyutsal normalizasyon)
# s sıfıra çok yakınsa sayısal dikkat gerekir; burada sadece şekli gösteriyoruz
res2 = res / s[:, None, None]
print('after divide shape', res2.shape, 'dtype', res2.dtype)

# Boyut sayısını raporla
print('res2 ndim', res2.ndim)

# Hızlı örnek: ilk frame’in ilk ekleminden (x,y) ilk 2 değer
print('Res2 first element sample:', res2[0,0,:4])


file C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\10_xy.npy
xy shape (210, 17, 2) dtype float32
xy_17 shape (210, 17, 2) ndim 3
left (210, 2) right (210, 2)
c (210, 2) s (210, 1)
c[:,None,:] (210, 1, 2)
s[:,None,None] (210, 1, 1, 1)
after subtract shape (210, 17, 2)
after divide shape (210, 210, 17, 2) dtype float32
res2 ndim 4
Res2 first element sample: [[-0.00978709 -1.0074695 ]
 [ 0.04215965 -1.1637623 ]
 [ 0.08086261 -1.1648535 ]
 [ 0.11329053 -1.1616557 ]]


# Adım 3 — TCN-Tabanlı Autoencoder (AE) Eğitimi ve Gömleme Çıkarma
Girdi: (B, C=40, T=30) → Encoder (TCN) → **embedding=64** → Decoder (TCN) → (B,40,30).
Kayıp: MSE (rekonstrüksiyon). En iyi model kaydedilir, sonra **train/test** için pencere gömlemeleri `.npy` olarak dışa aktarılır.


## 3.1 — Konfig, cihaz ve klasörler

In [ ]:
# Temel kütüphaneler ve PyTorch modülleri
import os, math, json, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader

# 2.7’de tanımladığımız değişkenleri kullanıyoruz:
# BASE_ROOT, WORK_DIR, STATS_JSON, CFG, train_loader, val_loader
# Ayrıca 2.7’deki yardımcıları (_make_features_CT, _window_slice_or_pad, _load_feature_stats) kullanacağız.
# NOT: Bu isimlerin bu hücreden önce tanımlı olması gerekir; burada sadece referans ediyoruz.

# Test dizinleri (Adım 2'de üretmiştik)
OUT_XY_TEST  = os.path.join(WORK_DIR, "npy_test_cikti")      # XY koordinat test çıktılarının bulunduğu klasör
OUT_ANG_TEST = os.path.join(WORK_DIR, "npy_aci_test_cikti")  # Açı tabanlı test çıktılarının bulunduğu klasör

# Model ve embedding çıktıları için klasörler (yoksa oluşturulur)
MODEL_DIR = os.path.join(WORK_DIR, "models"); os.makedirs(MODEL_DIR, exist_ok=True)             # AE/Checkpoint dosyaları
EMB_DIR_TRAIN = os.path.join(WORK_DIR, "embeddings", "train"); os.makedirs(EMB_DIR_TRAIN, exist_ok=True)  # Eğitim gömüleri
EMB_DIR_TEST  = os.path.join(WORK_DIR, "embeddings", "test");  os.makedirs(EMB_DIR_TEST,  exist_ok=True)  # Test gömüleri

# AE hiperparametrelerini CFG sözlüğüne "varsayılan" olarak ekle.
# setdefault: Anahtar yoksa verilen değeri atar; varsa mevcut değeri KORUR (üzerine yazmaz).
CFG.setdefault("emb_dim", 64)              # Latent boyutu
CFG.setdefault("hidden", 128)              # Gizli katman genişliği 
CFG.setdefault("lr", 1e-3)                 # Öğrenme oranı
CFG.setdefault("weight_decay", 1e-4)       # düzenlileştirme katsayısı
CFG.setdefault("max_epochs", 40)           # Maksimum epoch sayısı (erken durdurma tetiklenmezse)
CFG.setdefault("early_stop_patience", 6)   # Erken durdurma için : iyileşme yoksa kaç epoch sonra durdurulacak

# Donanım seçimi: GPU varsa 'cuda', yoksa 'cpu'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device  


device(type='cpu')

## 3.2 — Model: TCN Encoder/Decoder (residual, dilated)

In [12]:
# NOT: Bu hücrede sadece açıklama satırları eklendi; işlev/akış değiştirilmedi.

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1, k=3, p=None, dropout=0.1):
        super().__init__()
        # p parametresi verilmezse, zaman boyutunu (T) sabit tutacak "same" paddingi otomatik hesapla
        # k tek sayı olduğunda p = dilation * (k-1)//2, Conv1d çıkış uzunluğu girdiyle aynı kalır.
        if p is None: p = dilation * (k-1)//2  # length sabit kalsın

        # 1. konvolüsyon bloğu: kanal dönüşümü ve dilated conv ile alıcı alanı genişletme
        self.conv1 = nn.Conv1d(in_ch, out_ch, k, padding=p, dilation=dilation)
        self.bn1   = nn.BatchNorm1d(out_ch)    # dağılımı stabilize etmek için batch norm
        self.act1  = nn.GELU()                 # doğrusal olmayanlık (GELU)
        self.dropout = nn.Dropout(dropout)     # düzenlileştirme (overfit'i azaltmak için)

        # 2. konvolüsyon bloğu: aynı kanal sayısıyla bir kez daha özellik çıkarımı
        self.conv2 = nn.Conv1d(out_ch, out_ch, k, padding=p, dilation=dilation)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.act2  = nn.GELU()

        # Skip connection için projeksiyon: giriş/çıkış kanal sayısı farklıysa 1x1 conv ile eşitle
        # Aynıysa hiçbir işlem yapma (Identity)
        self.proj  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x şekli: (B,C,T)  B: batch, C: kanal sayısı, T: zaman uzunluğu
        y = self.conv1(x); y = self.bn1(y); y = self.act1(y); y = self.dropout(y)  # ilk yol
        y = self.conv2(y); y = self.bn2(y)                                         # ikinci yol (aktivasyon toplama sonrası)
        return self.act2(y + self.proj(x))  # residual toplama + aktivasyon (GELU)


class TCNEncoder(nn.Module):
    def __init__(self, in_ch=40, hidden=128, emb_dim=64, dilations=(1,2,4,8)):
        super().__init__()
        layers = []
        ch = in_ch
        # Artan dilations ile ResBlock zinciri kur (1,2,4,8): alıcı alan üstel artar
        for d in dilations:
            layers.append(ResBlock1D(ch, hidden, dilation=d))
            ch = hidden
        self.net = nn.Sequential(*layers)     # zaman boyutunda özellik çıkarımı yapan TCN gövdesi
        self.head = nn.Conv1d(hidden, hidden, 1)  # 1x1 conv: kanal içi harmanlama/projeksiyon
        self.pool = nn.AdaptiveAvgPool1d(1)   # (B,hidden,1) -> global ortalama havuzlama
        self.to_emb = nn.Linear(hidden, emb_dim)  # (B,hidden) -> (B,emb_dim) embedding katmanı

    def forward(self, x):
        # x: (B,40,T)  -> 40: giriş kanal/özellik sayısı (ör. 34 XY + 6 açı gibi)
        h = self.net(x)                # (B,hidden,T)  zaman ekseninde zengin özellikler
        h2 = self.head(h)              # (B,hidden,T)  1x1 ile yeniden karışım/projeksiyon
        g = self.pool(h2).squeeze(-1)  # (B,hidden)    global özet (temsil) vektörü
        z = self.to_emb(g)             # (B,emb_dim)   sabit boyutlu gömü (latent) vektörü
        return h2, z                   # zaman-özellik haritası ve embedding birlikte döndürülür


class TCNDecoder(nn.Module):
    def __init__(self, out_ch=40, hidden=128, dilations=(8,4,2,1)):
        super().__init__()
        layers = []
        ch = hidden
        # Encoder'ın ayna yapısı gibi ters sıra dilations (8,4,2,1) ile yeniden inşa bloğu
        for d in dilations:
            layers.append(ResBlock1D(ch, hidden, dilation=d))
            ch = hidden
        self.net = nn.Sequential(*layers)     # zaman-özellikten tekrar sinyale dönüşüm gövdesi
        self.out = nn.Conv1d(hidden, out_ch, 1)  # son katman: hedef kanal sayısına indirgeme

    def forward(self, h_time):
        # h_time: (B,hidden,T)  -> encoder'dan gelen zaman-özellik haritası
        y = self.net(h_time)
        return self.out(y)  # (B,40,T)  -> yeniden inşa edilen giriş sinyali


class AE_TCN(nn.Module):
    def __init__(self, in_ch=40, hidden=128, emb_dim=64):
        super().__init__()
        # Kodlayıcı: girişten zaman-özellik haritası ve embedding üretir
        self.encoder = TCNEncoder(in_ch=in_ch, hidden=hidden, emb_dim=emb_dim)
        # Çözücü: zaman-özellik haritasından girdiyi tekrar üretir
        self.decoder = TCNDecoder(out_ch=in_ch, hidden=hidden)

    @torch.no_grad()
    def encode(self, x):
        # x: (B,40,T)  -> sadece embedding gerektiğinde; gradyan hesabı kapalı
        h_time, z = self.encoder(x)
        return z

    def forward(self, x):
        # Tam autoencoder geçişi: yeniden inşa (x_hat) ve embedding (z) üret
        h_time, z = self.encoder(x)
        x_hat = self.decoder(h_time)
        return x_hat, z


## 3.3 — Eğitim döngüsü (early stopping + en iyi modeli kaydet)

In [13]:
# NOT: Sadece açıklama satırları eklendi; işlev/akış DEĞİŞTİRİLMEDİ.

def train_ae(model, train_loader, val_loader, cfg, model_dir):
    # Modeli uygun cihaza taşı (GPU varsa cuda, yoksa cpu)
    model = model.to(device)

    # AdamW optimizer: weight_decay ile L2 düzenlileştirme içerir
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    # Yeniden inşa kaybı için Ortalama Kare Hatası (MSE)
    crit = nn.MSELoss()

    # Erken durdurma ve en iyi model takibi için değişkenler
    best_val = float("inf")          # şimdiye kadarki en düşük val kaybı
    patience = 0                     # iyileşme olmadan geçen epoch sayısı
    best_path = os.path.join(model_dir, "best_ae_tcn.pt")  # en iyi checkpoint kaydı

    # Ana eğitim döngüsü
    for ep in range(1, cfg["max_epochs"]+1):
        model.train()                # eğitim moduna geç
        tr_loss, ntr = 0.0, 0        # epoch içi train toplam kayıp ve örnek sayacı

        # ---- Eğitim aşaması ----
        for xb in train_loader:
            xb = xb.to(device)       # minibatch: (B,40,30) bekleniyor
            x_hat, _ = model(xb)     # AE ileri geçiş: yeniden inşa ve embedding
            loss = crit(x_hat, xb)   # MSE yeniden inşa kaybı

            opt.zero_grad()          # önceki gradyanları temizle
            loss.backward()          # geriye yayılım
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradyan kırpma (stabilite)
            opt.step()               # ağırlıkları güncelle

            # Toplam kaybı ve örnek adedini topla (ağırlıklı ortalama için)
            tr_loss += loss.item() * xb.size(0); ntr += xb.size(0)

        # ---- Doğrulama aşaması ----
        model.eval()                 # değerlendirme moduna geç (dropout/bn davranışı)
        va_loss, nva = 0.0, 0
        with torch.no_grad():        # val'de gradyan hesabı gereksiz
            for xb in val_loader:
                xb = xb.to(device)
                x_hat, _ = model(xb)
                loss = crit(x_hat, xb)
                va_loss += loss.item() * xb.size(0); nva += xb.size(0)

        # Epoch ortalama kayıpları (örnek sayısına göre)
        tr = tr_loss / max(1,ntr)
        va = va_loss / max(1,nva)
        print(f"Epoch {ep:02d}/{cfg['max_epochs']}  train={tr:.6f}  val={va:.6f}")

        # ---- En iyi model kontrolü ve erken durdurma ----
        if va < best_val - 1e-6:
            # Val kaybında anlamlı iyileşme var -> en iyi değeri güncelle ve modeli kaydet
            best_val = va
            patience = 0
            torch.save({"state_dict": model.state_dict(),
                        "cfg": cfg}, best_path)
            print("  ↳ best updated & saved:", best_path)
        else:
            # İyileşme yok -> sabır sayacını artır
            patience += 1
            if patience >= cfg["early_stop_patience"]:
                # Belirlenen sabır eşiği aşıldı -> erken durdur
                print("  ↳ early stop.")
                break

    # Eğitim sonunda: diskte en iyi checkpoint varsa onu geri yükle
    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=device)
        model.load_state_dict(ckpt["state_dict"])

    # Eğitilmiş (ve en iyi haline geri yüklenmiş) modeli ve checkpoint yolunu döndür
    return model, best_path


## 3.4 — Eğitimi başlat

In [14]:

# AE_TCN modelini oluştur:
# - in_ch=40: giriş kanal sayısı (ör. 34 XY + 6 açı gibi)
# - hidden: TCN bloklarının kanal genişliği (CFG'den)
# - emb_dim: çıkarılacak latent/embedding boyutu (CFG'den)
model = AE_TCN(in_ch=40, hidden=CFG["hidden"], emb_dim=CFG["emb_dim"])

# Modeli eğit:
# - train_loader / val_loader: eğitim ve doğrulama minibatçları
# - CFG: hiperparametreler (lr, weight_decay, max_epochs, early_stop_patience, vs.)
# - MODEL_DIR: en iyi checkpoint'in kaydedileceği klasör
model, BEST_PATH = train_ae(model, train_loader, val_loader, CFG, MODEL_DIR)

# val kaybı en düşük yani en iyi  modelin checkpoint yolunu yazdır
print("Best model:", BEST_PATH)


Epoch 01/40  train=1.310158  val=0.635254
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 02/40  train=0.504361  val=0.585095
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 03/40  train=0.298549  val=0.391577
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 04/40  train=0.209853  val=0.244242
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 05/40  train=0.180577  val=0.158148
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 06/40  train=0.145542  val=0.130389
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 07/40  train=0.138315  val=0.115459
  ↳ best updated & saved: C:

## 3.5 — Gömleme dışa aktarma (train/test → .npy)

In [18]:

from pathlib import Path
import glob

@torch.no_grad()
def export_embeddings(model, xy_dir, ang_dir, out_dir, cfg, file_list=None, stats_json=STATS_JSON):
    # Çıktı klasörünü oluştur (varsa dokunma)
    os.makedirs(out_dir, exist_ok=True)

    # Özellik normalizasyonu için ortalama ve std değerlerini yükle
    mean, std = _load_feature_stats(stats_json)

    # XY kaynak dosyalarını topla
    xy_files = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
    # İsteğe bağlı: belirli bir dosya listesi verilmişse, yalnızca onları kullan
    if file_list is not None:
        xy_files = [os.path.join(xy_dir, f"{b}_xy.npy") for b in file_list]
    # En az bir dosya olmalı
    assert xy_files, f"Boş dizin: {xy_dir}"

    # Pencereleme ve batch ayarları (cfg'den)
    win, stride, pad = cfg["window_len"], cfg["stride"], cfg["pad_mode"]
    bs = cfg["batch_size"]
    saved = []

    # Modeli eval moduna al (dropout/bn inference davranışı)
    model.eval()

    # Her dosya için embedding çıkar
    for p in xy_files:
        base = Path(p).name.replace("_xy.npy","")          # taban ad (örn. 001, 61test vs.)
        pang = os.path.join(ang_dir, f"{base}_ang.npy")    # açı dosya yolu
        xy = np.load(p)                                    # XY: (kanal_x2, T) ya da (40,T) için girdi seti
        ang = np.load(pang) if os.path.exists(pang) else None  # açı varsa yükle, yoksa None

        # XY + (opsiyonel) açıları birleştirerek nihai özellik vektörü üret (40,T)
        feat = _make_features_CT(xy, ang, cfg["use_angles"])   # (40,T)
        T = feat.shape[1]                                      # toplam zaman uzunluğu

        # Pencere başlangıç indeksleri (overlap = stride)
        starts = [0] if T <= win else list(range(0, T - win + 1, stride))
        # Sona taşan kısım kalırsa son pencereyi T-win'e sabitle
        if T > win and (T - win) % stride != 0:
            starts.append(T - win)

        # Mini-batch halinde encode işlemi
        embs = []
        for i in range(0, len(starts), bs):
            batch_starts = starts[i:i+bs]
            x_list = []
            for st in batch_starts:
                # Pencereleri dilimle ya da pad et (cfg["pad_mode"]'a göre)
                xw = _window_slice_or_pad(feat, st, win, pad)        # (40,win)
                # Kanal bazlı standardizasyon: (x - mean) / std
                xw = (xw - mean[:,None]) / (std[:,None] + 1e-8)      # sayısal stabilite için 1e-8
                x_list.append(xw)
            # Batch eksenini ekle
            x = np.stack(x_list, axis=0)                              # (B,40,win)
            x = torch.from_numpy(x).float().to(device)                # tensöre çevir ve cihaza taşı
            z = model.encode(x)                                       # (B,emb_dim) sadece encoder embedding'i
            embs.append(z.cpu().numpy())                              # CPU'ya al ve numpy'a çevir

        # Tüm pencerelerin embedding'lerini birleştir
        E = np.concatenate(embs, axis=0) if embs else np.zeros((0, cfg["emb_dim"]), np.float32)
        # Dosya bazında embedding'i kaydet
        out_path = os.path.join(out_dir, f"{base}_emb.npy")
        np.save(out_path, E.astype(np.float32))
        # Kayıt özeti: (taban_ad, şekil)
        saved.append((base, E.shape))
    return saved

# ---- Eğitim seti için embedding çıkar ve kaydet ----
train_saved = export_embeddings(model,
                                xy_dir=os.path.join(WORK_DIR, "npy_cikti"),        # eğitim XY kaynakları
                                ang_dir=os.path.join(WORK_DIR, "npy_aci_cikti"),   # eğitim açı kaynakları
                                out_dir=EMB_DIR_TRAIN,                              # çıktı: embeddings/train
                                cfg=CFG)                                            # cfg: window_len/stride/pad_mode/batch_size/emb_dim
print("Train embeddings saved:", len(train_saved), "dosya")

# ---- Test seti için embedding çıkar ve kaydet (örn. 61,62,63,64) ----
test_saved = export_embeddings(model,
                               xy_dir=OUT_XY_TEST,                                  # test XY kaynakları
                               ang_dir=OUT_ANG_TEST,                                # test açı kaynakları
                               out_dir=EMB_DIR_TEST,                                # çıktı: embeddings/test
                               cfg=CFG)
print("Test embeddings saved:", len(test_saved), "dosya")


Train embeddings saved: 60 dosya
Test embeddings saved: 3 dosya


# Adım 4 — DTW-yol Ortalama Cosine (%) ile Benzerlik
Karşılaştırmalar:
- 61 ↔ 61 (kontrol)
- 61 ↔ 62 (farklı hareket)
- 61 ↔ 63 (aynı hareket, **2× hız**) 


## 4.1 — Yardımcılar: gömleme yükleme + DTW yol ve skor

In [19]:

import os, numpy as np

# Gömleme klasörleri (Adım 3'te oluşturulmuş olmalı)
EMB_DIR_TEST  = os.path.join(WORK_DIR, "embeddings", "test")
EMB_DIR_TRAIN = os.path.join(WORK_DIR, "embeddings", "train")  # gerekirse eğitim emb'leri

# fastdtw varsa kullan, yoksa klasik DTW (path’li) kullan
try:
    from fastdtw import fastdtw
    _HAVE_FASTDTW = True
except Exception:
    _HAVE_FASTDTW = False

def load_emb(path):
    """
    .npy gömleme dosyasını (T, D) biçiminde yükler ve float32'ye çevirir.
    T: pencere sayısı (Nwin), D: embedding boyutu.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Bulunamadı: {path}")
    arr = np.load(path)                          # dosyayı yükle
    arr = np.asarray(arr, dtype=np.float32)      # tip güvenliği: float32
    if arr.ndim != 2:                            # beklenen: (T, D)
        arr = arr.reshape(arr.shape[0], -1)      # değilse 2D'ye sıkıştır
    return arr                                   # (T, D) = (Nwin, emb_dim)

def _cosine_distance(u: np.ndarray, v: np.ndarray, eps: float=1e-8) -> float:
    """
    Kosinüs uzaklığı = 1 - cosine_similarity(u, v)
    Bölümde 0 hatasını önlemek için küçük eps eklenir.
    """
    uu = float(np.linalg.norm(u) + eps)          # ||u|| + eps
    vv = float(np.linalg.norm(v) + eps)          # ||v|| + eps
    return 1.0 - float(np.dot(u, v) / (uu * vv)) # 1 - (u·v / (||u||·||v||))

def _dtw_path_exact(A: np.ndarray, B: np.ndarray):
    """
    Klasik (tam) DTW hesaplaması: maliyet matrisi üzerinden en iyi eşleme yolunu döndürür.
    Zaman karmaşıklığı ~ O(T1*T2). Path geri izlenerek elde edilir.
    """
    T1, D1 = A.shape; T2, D2 = B.shape
    assert D1 == D2, f"embed boyutu farklı: {D1} vs {D2}"
    # Kümülatif maliyet matrisi (bir hücre fazlalıklı sınırlarla)
    C = np.full((T1+1, T2+1), np.inf, dtype=np.float64)
    C[0,0] = 0.0
    # Geri izleme için yön matrisi (üst/sol/çapraz)
    back = np.zeros((T1+1, T2+1, 2), dtype=np.int32) - 1
    for i in range(1, T1+1):
        for j in range(1, T2+1):
            cost = _cosine_distance(A[i-1], B[j-1])  # yerel eşleme maliyeti
            # üç komşudan (üst, sol, çapraz) en düşüğü
            k = np.argmin((C[i-1,j], C[i,j-1], C[i-1,j-1]))
            if k == 0:   C[i,j] = cost + C[i-1,j];   back[i,j] = (i-1, j)
            elif k == 1: C[i,j] = cost + C[i, j-1];  back[i,j] = (i, j-1)
            else:        C[i,j] = cost + C[i-1,j-1]; back[i,j] = (i-1, j-1)
    # Geri izleme: (T1, T2)'den (0,0)'a
    path = []
    i, j = T1, T2
    while not (i == 0 and j == 0):
        path.append((i-1, j-1))
        i, j = back[i,j]
    path.reverse()                                 # ileri sıraya çevir
    return path

def dtw_mean_cosine_percent(E1: np.ndarray, E2: np.ndarray) -> float:
    """
    DTW yoluna göre ortalama kosinüs benzerliği (yüzde) hesaplar.
    Girdi:
      - E1, E2: (T, D) gömleme dizileri
    Çıktı:
      - Ortalama cosine similarity * 100 (yüzde)
    """
    E1 = np.asarray(E1, dtype=np.float32)
    E2 = np.asarray(E2, dtype=np.float32)
    assert E1.ndim == 2 and E2.ndim == 2
    assert E1.shape[1] == E2.shape[1], f"embed boyutu uymuyor: {E1.shape[1]} vs {E2.shape[1]}"

    if _HAVE_FASTDTW:
        # fastdtw, verilen mesafe fonksiyonuna göre (maliyet, path) döndürür
        _, path = fastdtw(E1, E2, dist=_cosine_distance)
    else:
        # Tam DTW (daha yavaş ama kesin) yol
        path = _dtw_path_exact(E1, E2)

    # Yol üzerindeki tüm eşleşmeler için cosine similarity topla
    sims = []
    for i, j in path:
        sims.append(1.0 - _cosine_distance(E1[i], E2[j]))  # 1 - dist = similarity
    mean_sim = float(np.mean(sims)) if sims else 0.0       # ortalama benzerlik
    return 100.0 * mean_sim                                # yüzde olarak döndür


## 4.2 — 61/62/63 karşılaştırmaları

In [20]:

# ---- Dosya yolları (test embedding'leri) ----
p61 = os.path.join(EMB_DIR_TEST, "61_emb.npy")  # 61 numaralı test videonun embedding'i
p62 = os.path.join(EMB_DIR_TEST, "62_emb.npy")  # 62 numaralı test videonun embedding'i
p63 = os.path.join(EMB_DIR_TEST, "63_emb.npy")  # 63: 61'in 2× hızlandırılmış versiyonu

# ---- Embedding'leri yükle ----
E61 = load_emb(p61)  # (T1, D)
E62 = load_emb(p62)  # (T2, D)
E63 = load_emb(p63)  # (T3, D)

# ---- DTW + kosinüs benzerliği ile skorlar (yüzde) ----
sim_61_61 = dtw_mean_cosine_percent(E61, E61)   # kontrol: aynı sinyal -> yüksek beklenir
sim_61_62 = dtw_mean_cosine_percent(E61, E62)   # farklı hareket -> düşük/orta beklenir
sim_61_63 = dtw_mean_cosine_percent(E61, E63)   # aynı hareket (2× hız) -> yüksek beklenir (DTW hizalar)

# ---- Sonuçları yazdır ----
print(f"61 vs 61  (kontrol)   : {sim_61_61:6.2f} %")
print(f"61 vs 62  (farklı)    : {sim_61_62:6.2f} %")
print(f"61 vs 63  (aynı, 2×)  : {sim_61_63:6.2f} %")


61 vs 61  (kontrol)   : 100.00 %
61 vs 62  (farklı)    :  52.46 %
61 vs 63  (aynı, 2×)  :  94.90 %


In [21]:
# NOT: Sadece açıklama satırları eklendi; işlev/akış DEĞİŞTİRİLMEDİ.

import os, numpy as np, torch, torch.nn as nn

def recon_mse_for_video(vid):
    # İlgili test videonun XY ve (varsa) açı dosya yollarını kur
    xy_p  = os.path.join(WORK_DIR, "npy_test_cikti", f"{vid}_xy.npy")
    ang_p = os.path.join(WORK_DIR, "npy_aci_test_cikti", f"{vid}_ang.npy")

    # NPY dosyalarını yükle (açı dosyası yoksa None)
    xy = np.load(xy_p); ang = np.load(ang_p) if os.path.exists(ang_p) else None

    # XY + (opsiyonel) açıları birleştirerek (40, T) özellik tensörünü üret
    feat = _make_features_CT(xy, ang, CFG["use_angles"])                    # (40,T)

    # Normalizasyon istatistikleri ve pencereleme ayarlarını al
    mean, std = _load_feature_stats(STATS_JSON)
    win, stride, pad = CFG["window_len"], CFG["stride"], CFG["pad_mode"]

    # --- Pencereleme: tüm başlangıç indekslerini hesapla ---
    T = feat.shape[1]
    starts = [0] if T<=win else list(range(0, T-win+1, stride))  # tam sığarsa tek pencere
    if T>win and (T-win)%stride!=0: starts.append(T-win)         # artan kısım varsa son pencereyi ekle

    # Her pencere için dilimle/pad et, kanal bazlı standardize et ve listeye ekle
    X = []
    for st in starts:
        xw = _window_slice_or_pad(feat, st, win, pad)            # (40,win)
        xw = (xw - mean[:,None])/(std[:,None]+1e-8)              # standardizasyon (+eps stabilite için)
        X.append(xw)

    # (B,40,win) biçiminde batch tensörü oluştur ve cihaza taşı
    X = torch.from_numpy(np.stack(X)).float().to(device)         # (B,40,30)

    # --- Autoencoder ile yeniden inşa ve MSE ölçümü ---
    model.eval()
    with torch.no_grad():
        x_hat, _ = model(X)                                      # yeniden inşa ve embedding
        mse = nn.functional.mse_loss(x_hat, X, reduction="mean").item()  # ortalama MSE
    return mse

# Seçilen test videoları için yeniden inşa hatasını (MSE) yazdır
for v in [61,62,63]:
    print(v, "MSE:", recon_mse_for_video(v))


61 MSE: 0.09801687300205231
62 MSE: 61.76719284057617
63 MSE: 0.13490988314151764
